# 3. Procesamiento ETL: PUI (Plataforma Única de Identidad)

**Objetivo:** generar `output/Resultado_3_Procesamiento_ETL_PUI.parquet`, que consolida las
coincidencias administrativas de la PUI con las confrontas manuales del RNPDNO y con las
defunciones por institución. El flujo oficial lee RNPDNO desde el archivo autónomo; el pipeline
principal consume la salida autónoma de PUI configurada en `CONFRONTAS_FILE`.

## Insumos

| Objeto en R | Archivo | Columnas que se usan |
|---|---|---|
| `coincidencias_admin_con_inactivas` | `input/pui/reporte_coincidencias_validas_YYYY-MM-DD.csv` | las 16 que valida el script |
| `registros` | `input/pui/registros_con_pui_DDMMYYYY.xlsx` | `fub`, `curp_victima`, `indicadora_indicio_de_vida`, `institucion_indicadora_de_vida`, `numero_instituciones` |
| `defunciones_df` | `input/pui/tabla_defunciones_por_institucion_*.xlsx` | `fub`, `tipo_defuncion_institucion` |
| `comprimido` | `output/Resultado_1_Procesamiento_ETL_RNPDNO_autonomo.parquet` | `fub`, CURP derivada, clasificación y fecha de hechos modificada |

## Salida de este ETL oficial

`output/Resultado_3_Procesamiento_ETL_PUI.parquet` — 10 columnas:
`fub`, `curp_victima`, `indicadora_indicio_de_vida`, `institucion_indicadora_de_vida`,
`numero_instituciones`, `indicadora_coincidencia_manual`, `tipo_tramites`,
`tramites_repetidos`, `fecha_evento`, `clasificacion_des`.

La entrada que lee el pipeline principal es `output/Resultado_3_Procesamiento_ETL_PUI_autonomo.parquet`.

In [ ]:
import os
import re
import sys
import warnings
from collections import Counter
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

# Configuración de rutas (misma convención que los notebooks 1 y 2)
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import config

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

print(f"📁 Raíz del proyecto: {project_root}")

## El insumo de la PUI
De dónde sale el CSV de coincidencias lo decide **`config.MODO_INSUMOS`**, igual que en el notebook 1, y los dos caminos son excluyentes: el que no se eligió no se intenta nunca, ni como respaldo del otro.

| Modo | Qué hace | Si falla |
|---|---|---|
| `"drive"` (por omisión) | Baja el **zip diario** (`reporte_coincidencias_validas_YYYY-MM-DD.zip`), lo descomprime y deja el CSV en `input/pui/`. Si ese corte ya está en disco, no baja nada. | **Se detiene.** No corre con el CSV que haya quedado en disco. |
| `"local"` | Lee el archivo exacto de `config.PUI_LOCAL_FILE`. No toca Drive, no pide credenciales, no revisa la fecha del corte. | **Se detiene** diciendo qué ruta falta. |

Para correr sin descargar nada: cambia `MODO_INSUMOS = "local"` en `config.py` y apunta `PUI_LOCAL_FILE` a tu archivo (o lanza Jupyter con `PIPELINE_MODO_INSUMOS=local`). Ojo: el nombre del CSV debe seguir trayendo la fecha (`..._YYYY-MM-DD.csv`), porque de ahí sale la fecha de corte de la corrida.

**En modo `"drive"`, el corte tiene que ser el de hoy.** Si el más nuevo que hay en Drive no es del día, el proceso se detiene con `NO SE TIENE EL ARCHIVO DEL DÍA` en vez de dejar que el reporte salga calculado sobre datos viejos. Es la misma regla del notebook 1, con su propia tolerancia (`GDRIVE_PUI_DIAS_TOLERANCIA`).

Las credenciales son las mismas de siempre, en `.secrets/`. Si es la primera vez, sigue `docs/9_credenciales_google_drive.md`.

In [ ]:
# El insumo de la PUI: se baja de Drive o se lee de la ruta local, según
# config.MODO_INSUMOS (ver src/drive.py). Los dos modos son excluyentes.
import importlib

from src import drive
importlib.reload(drive)

print(f"⚙️  Modo de insumos: {config.PUI_MODO_INSUMOS!r}\n")
drive.sincronizar_pui()

# `config.PUI_COINCIDENCIAS_FILE` se resuelve al importar config, así que hay que
# recargarlo para que apunte al CSV recién bajado.
importlib.reload(config)

print(f"\n📄 Insumo de la PUI: {config.PUI_COINCIDENCIAS_FILE}")
print(f"   Corte declarado en el nombre: "
      f"{config.fecha_corte_pui(config.PUI_COINCIDENCIAS_FILE)}")

## Parámetros: rutas de los insumos

Por defecto se toma el archivo **más reciente** de cada patrón. Para fijar un archivo concreto
—la convención de control estricto que usa el resto del pipeline— sustituye el valor por una
ruta explícita.

In [ ]:
from src.validacion import InsumoInvalido, leer_csv, leer_excel, leer_parquet

# ── Insumos: las etapas previas se leen desde sus salidas autónomas ──────────
# Los insumos manuales (registros, defunciones) llevan control estricto. Este
# notebook oficial mantiene su destino de escritura ETL_PUI_OUTPUT_FILE; el
# pipeline principal consume ETL_PUI_INPUT_FILE (la salida autónoma de PUI).
RUTA_COINCIDENCIAS = config.PUI_COINCIDENCIAS_FILE
RUTA_REGISTROS     = config.PUI_REGISTROS_FILE
RUTA_DEFUNCIONES   = config.PUI_DEFUNCIONES_FILE
RUTA_COMPRIMIDO    = config.ETL_RNPDNO_INPUT_FILE
RUTA_SALIDA        = config.ETL_PUI_OUTPUT_FILE

INSUMOS = [
    ("coincidencias PUI",   RUTA_COINCIDENCIAS, "config.PUI_COINCIDENCIAS_FILE"),
    ("registros (manual)",  RUTA_REGISTROS,     "config.PUI_REGISTROS_FILE"),
    ("defunciones",         RUTA_DEFUNCIONES,   "config.PUI_DEFUNCIONES_FILE"),
    ("RNPDNO autónomo",     RUTA_COMPRIMIDO,    "config.ETL_RNPDNO_INPUT_FILE"),
]

# Se comprueban los cuatro de golpe, para no ir descubriendo faltantes de uno en uno.
faltan = [(e, r, c) for e, r, c in INSUMOS if not Path(r).exists()]
if faltan:
    detalle = "\n".join(f"     - {e}: {r}   ({c})" for e, r, c in faltan)
    extra = ("\n   Genera primero output/Resultado_1_Procesamiento_ETL_RNPDNO_autonomo.parquet."
             if any(c.endswith("ETL_RNPDNO_INPUT_FILE") for _, _, c in faltan) else "")
    raise InsumoInvalido(
        f"❌ Faltan {len(faltan)} de los {len(INSUMOS)} insumos del cuaderno:\n"
        f"{detalle}\n"
        f"   Corrige la ruta en config.py o coloca el archivo.{extra}"
    )

print("Insumos (declarados en config.py):")
for etiqueta, ruta, clave in INSUMOS:
    print(f"  ✅ {etiqueta:22s} {Path(ruta).name}")

# ── Fecha de corte ───────────────────────────────────────────────────────────
# Sólo etiqueta la corrida: el archivo de salida tiene nombre fijo. Si config no
# la fija, se deriva del nombre del CSV de la PUI.
def _fecha_corte(ruta_csv) -> str:
    if getattr(config, "PUI_FECHA_CORTE", None):
        return config.PUI_FECHA_CORTE
    m = re.search(r"(\d{4})-(\d{2})-(\d{2})", Path(ruta_csv).name)
    if not m:
        raise InsumoInvalido(
            f"❌ No se pudo deducir la fecha de corte de {Path(ruta_csv).name!r}.\n"
            f"   Se esperaba que el nombre incluyera YYYY-MM-DD. Fíjala a mano en "
            f"config.PUI_FECHA_CORTE (formato DDMMYYYY)."
        )
    anio, mes, dia = m.groups()
    return f"{dia}{mes}{anio}"

FECHA_CORTE = _fecha_corte(RUTA_COINCIDENCIAS)

print(f"\nCorte de los datos : {FECHA_CORTE}")
print(f"Salida de este ETL oficial: {RUTA_SALIDA}")
print(f"Entrada PUI del pipeline principal: {config.CONFRONTAS_FILE}")

## Paso 0. Equivalencias R → pandas

El script de R se apoya en verbos de `dplyr`/`stringr` cuyo comportamiento con nulos y con el
orden de los elementos no coincide con el de pandas por defecto. Estas funciones replican la
semántica exacta de R para que la salida sea idéntica.

In [ ]:
def clean_names(df: pd.DataFrame) -> pd.DataFrame:
    """Equivalente mínimo de janitor::clean_names() para columnas ya en snake_case."""
    df = df.copy()
    df.columns = [
        re.sub(r"_+", "_", re.sub(r"[^0-9a-zA-Z]+", "_", str(c).strip())).strip("_").lower()
        for c in df.columns
    ]
    return df


def primero(serie: pd.Series):
    """dplyr::first() — primer valor del grupo, incluido NA.

    Se distingue de pandas GroupBy.first(), que salta los nulos.
    """
    return serie.iloc[0] if len(serie) else np.nan


def paste_unique(serie: pd.Series, sep: str = ", ") -> str:
    """paste(unique(x), collapse = sep) — conserva el orden de primera aparición."""
    return sep.join(dict.fromkeys(serie.astype(str)))


def paste_sorted_unique(serie: pd.Series, sep: str = ", ") -> str:
    """paste(sort(unique(x)), collapse = sep)."""
    return sep.join(sorted(dict.fromkeys(serie.astype(str))))


def r_str_replace_all(serie: pd.Series, reemplazos: dict) -> pd.Series:
    """stringr::str_replace_all() con un vector nombrado.

    Aplica los patrones **en secuencia**, no en paralelo: la salida de un
    reemplazo es la entrada del siguiente. Es lo que hace R, y aquí importa —
    "Secretaría de Infraestructura, Comunicaciones y Transportes (SICT)" pasa por
    dos reglas y termina como "SICT, SICT".
    """
    out = serie
    for patron, valor in reemplazos.items():
        out = out.str.replace(patron, valor, regex=True)
    return out


def conteos_en_orden_de_aparicion(texto: str) -> str:
    """Réplica de:  conteos <- table(x); paste(as.integer(conteos[unique(x)]), collapse = ", ")"""
    if not isinstance(texto, str):
        return np.nan
    partes = texto.split(", ")
    conteos = Counter(partes)
    return ", ".join(str(conteos[p]) for p in dict.fromkeys(partes))


def unicos_en_orden_de_aparicion(texto: str) -> str:
    """Réplica de:  unique(str_split(x, ", ")) |> paste(collapse = ", ")"""
    if not isinstance(texto, str):
        return np.nan
    return ", ".join(dict.fromkeys(texto.split(", ")))


def comma(n) -> str:
    """scales::comma()."""
    return f"{n:,}"


# Diccionario de abreviaturas de instituciones. El script de R lo repite tres veces
# (dos sobre `instituciones`, una sobre `tipo_tramites`); aquí se define una sola vez.
# El orden de las claves es significativo: ver r_str_replace_all().
ABREVIATURAS_INSTITUCIONES = {
    r"Comisión Nacional del Sistema de Ahorro para el Retiro \(CONSAR\)": "CONSAR",
    r"Comunicaciones y Transportes \(SICT\)": "SICT",
    r"INSTITUTO MEXICANO DEL SEGURO SOCIAL": "IMSS-PUI",
    r"Instituto Nacional de Migración \(INM\)": "INAMI",
    r"Petróleos Mexicanos \(PEMEX\)": "PEMEX",
    r"Secretaría de Bienestar \(SEBIEN\)": "SEBIEN",
    r"Secretaría de Infraestructura": "SICT",
    r"Secretaría de Relaciones Exteriores \(SRE\)": "SRE",
    r"Servicio de Administración Tributaria \(SAT\)": "SAT",
    r"Servicios de Salud del Instituto Mexicano del Seguro Social para el Bienestar \(IMSS-Bienestar\)": "IMSS-Bienestar",
}

print("✅ Helpers de equivalencia R → pandas definidos.")

## Paso 1. Carga del CSV de la PUI

Réplica de `leer_csv_pui()`: se intenta UTF-8 y, si falla, Latin-1. Todas las columnas se leen
como texto para preservar los ceros a la izquierda de `institucion_id` (`"0044"` = RENAPO), que
el script usa como filtro.

In [ ]:
# Lectura con validación: exige el archivo y las 16 columnas de
# config.PUI_COINCIDENCIAS_COLUMN_MAPPING. Si falta alguna, se detiene aquí.
#
# `solo_mapeadas=False` conserva cualquier columna extra que traiga el archivo:
# el mapeo fija el MÍNIMO exigido, no el máximo permitido.
#
# Todo se lee como texto para preservar los ceros a la izquierda de
# `institucion_id` ("0044" = RENAPO), que el cuaderno usa como filtro.
coincidencias = leer_csv(
    RUTA_COINCIDENCIAS,
    config.PUI_COINCIDENCIAS_COLUMN_MAPPING,
    descripcion="coincidencias de la PUI",
    solo_mapeadas=False,
    dtype=str, low_memory=False,
)
coincidencias = clean_names(coincidencias)

print(f"\u2705 Coincidencias de la PUI cargadas: {comma(len(coincidencias))} filas, "
      f"{coincidencias.shape[1]} columnas.")
coincidencias.head(3)

## Paso 2. Validaciones de entrada

Las tres comprobaciones del script: columnas esperadas (error), fases de búsqueda válidas
(error) y CURP vacías (advertencia).

In [ ]:
# Las columnas ya se exigieron al leer el archivo, contra
# config.PUI_COINCIDENCIAS_COLUMN_MAPPING: si faltara alguna, el cuaderno se
# habría detenido en la celda anterior. Aquí sólo se deja constancia.
COLUMNAS_ESPERADAS = list(config.PUI_COINCIDENCIAS_COLUMN_MAPPING)
print(f"¡Perfecto! El archivo contiene las {len(COLUMNAS_ESPERADAS)} columnas exigidas "
      f"por config.PUI_COINCIDENCIAS_COLUMN_MAPPING.\n")

# --- Fases de búsqueda: sólo 2 y 3 son válidas ---
fase_num = pd.to_numeric(coincidencias["fase_busqueda"], errors="coerce")
if (~fase_num.isin([2, 3])).any():
    invalidas = coincidencias.loc[~fase_num.isin([2, 3]), "fase_busqueda"].value_counts()
    raise ValueError(
        "¡Atención! Se detectaron fases de búsqueda no válidas en el archivo de "
        f"coincidencias administrativas. Revisa los datos para identificar la discrepancia.\n"
        f"Valores encontrados:\n{invalidas}"
    )
print("¡Perfecto! Todas las fases de búsqueda en el archivo de coincidencias administrativas son válidas.\n")

# --- CURP vacías: sólo advertencia, el script no detiene el proceso ---
n_curp_vacias = int(coincidencias["curp"].isna().sum())
if n_curp_vacias > 0:
    warnings.warn(
        f"¡Atención! Se detectaron {comma(n_curp_vacias)} curps vacías en la base "
        "coincidencias_admin_con_inactivas. Revisa los datos para identificar la discrepancia."
    )
else:
    print("¡Perfecto! No se detectaron curps vacías en la base coincidencias_admin_con_inactivas.\n")

# `is_valid` llega como texto ("True"/"False"); el script lo trata como lógico.
coincidencias["is_valid"] = coincidencias["is_valid"].map(
    {"True": True, "true": True, "TRUE": True, "t": True,
     "False": False, "false": False, "FALSE": False, "f": False}
).astype("boolean")

if coincidencias["is_valid"].isna().any():
    warnings.warn("Hay valores de `is_valid` que no se pudieron interpretar como lógicos.")

print(f"is_valid → {coincidencias['is_valid'].value_counts(dropna=False).to_dict()}")

## Paso 3. Limpieza de las coincidencias

1. **FUB sin UUID.** La columna `fub` del CSV concatena el FUB del RNPDNO con un UUID de la
   coincidencia. El script recorta el UUID de 32 hexadecimales del final y el guion sobrante.
2. **CURP inválidas conocidas.** Se eliminan las listadas en `input/pui/curps_excluidas.txt`,
   una por renglón. Viven fuera de git porque son datos personales.
3. **RENAPO (`institucion_id == "0044"`).** Se descartan las actas de nacimiento y los eventos
   que contienen `|1`. Desde el 8 de junio de 2026 el resto de RENAPO sí se conserva.
4. **Institución en el evento.** A `tipo_evento` se le adjunta la institución entre paréntesis.

In [ ]:
antes = len(coincidencias)

# 1. FUB: quitar el UUID de la coincidencia que va pegado al final.
#    R: str_remove(fub_uuid, "([0-9A-F]-?){32}-?$") y luego str_remove(fub, "-$")
coincidencias = coincidencias.rename(columns={"fub": "fub_uuid"})
coincidencias["fub"] = (
    coincidencias["fub_uuid"]
      .str.replace(r"([0-9A-F]-?){32}-?$", "", regex=True)
      .str.replace(r"-$", "", regex=True)
)
print("Ejemplo de recorte del FUB:")
print(coincidencias[["fub_uuid", "fub"]].head(3).to_string(index=False), "\n")

# 2. CURP inválidas conocidas (lista en input/pui/, fuera de git). En R, `curp != "..."`
#    con NA descarta la fila; se replica aplicando el mismo filtro por cada CURP.
if config.PUI_CURPS_EXCLUIDAS_FILE.exists():
    for curp in config.PUI_CURPS_EXCLUIDAS_FILE.read_text(encoding="utf-8").split():
        coincidencias = coincidencias[coincidencias["curp"].ne(curp).fillna(False)]
else:
    warnings.warn(f"No existe {config.PUI_CURPS_EXCLUIDAS_FILE}: no se excluye ninguna CURP.")

# 3. RENAPO: fuera actas de nacimiento y eventos con "|1".
#    A & B con B nulo da NA en R, y filter(!NA) descarta: por eso na=True en el contains.
es_renapo = coincidencias["institucion_id"].eq("0044").fillna(False)

es_acta_nacimiento = coincidencias["tipo_evento"].str.contains(
    "Tipo de Acto: Nacimiento", regex=False, na=True)
warnings.warn(
    "¡Atención! Se eliminarán las coincidencias administrativas provenientes de la institución "
    "RENAPO (institucion_id == '0044') y que sean actas de nacimiento. Revisa los datos para "
    "identificar la discrepancia."
)
coincidencias = coincidencias[~(es_renapo & es_acta_nacimiento)]

es_renapo = coincidencias["institucion_id"].eq("0044").fillna(False)
tiene_pipe_uno = coincidencias["tipo_evento"].str.contains(r"\|1", regex=True, na=True)
coincidencias = coincidencias[~(es_renapo & tiene_pipe_uno)]

# 4. Adjuntar la institución al tipo de evento. paste0() convierte NA en el texto "NA".
coincidencias["tipo_evento"] = (
    coincidencias["tipo_evento"].fillna("NA") + " (" + coincidencias["institucion"].fillna("NA") + ")"
)

print(f"Filas antes de la limpieza:  {comma(antes)}")
print(f"Filas después de la limpieza: {comma(len(coincidencias))}  "
      f"(se descartaron {comma(antes - len(coincidencias))})")

## Paso 4. Agregación por FUB

Una fila por FUB. `first()` de dplyr toma el primer valor del grupo aunque sea nulo —a
diferencia de `GroupBy.first()` de pandas, que lo salta—, por eso se usa el helper `primero`.

In [ ]:
coincidencias_agrupado = (
    coincidencias
      .groupby("fub", sort=True, dropna=False)
      .agg(
          no_apariciones      = ("fub", "size"),
          curp                = ("curp", primero),
          is_valid            = ("is_valid", primero),  
          instituciones       = ("institucion", paste_unique),
          tipo_evento         = ("tipo_evento", lambda s: ", ".join(s.astype(str))),
          no_instituciones    = ("institucion_id", lambda s: s.nunique(dropna=False)),
          fases_busqueda      = ("fase_busqueda", paste_sorted_unique),
          instituciones_ids   = ("institucion_id", paste_unique),
          fecha_desaparicion  = ("fecha_desaparicion", primero),
          fecha_evento        = ("fecha_evento", primero),
          calle               = ("calle", primero),
          colonia             = ("colonia", primero),
          codigo_postal       = ("codigo_postal", primero),
          direccion           = ("direccion", primero),
      )
      .reset_index()
      .rename(columns={"is_valid": "desaparicion_activa"})
)

print(f"✅ Coincidencias agrupadas por FUB: {comma(len(coincidencias_agrupado))} filas.")
coincidencias_agrupado.head(3)

### Conteo de trámites repetidos

`tipo_evento` se parte por `", "`. **Los propios eventos contienen comas** (`"Fecha Registro:
1967-09-23, CURP: ..., Tipo de Acto: Nacimiento, ..."`), de modo que el corte no separa eventos
completos sino fragmentos. Es el comportamiento del script de R y se conserva tal cual para no
alterar la salida.

In [ ]:
coincidencias_agrupado["tramites_repetidos"] = (
    coincidencias_agrupado["tipo_evento"].map(conteos_en_orden_de_aparicion)
)
coincidencias_agrupado["tipo_evento"] = (
    coincidencias_agrupado["tipo_evento"].map(unicos_en_orden_de_aparicion)
)

num_activas   = int((coincidencias_agrupado["desaparicion_activa"] == True).sum())
num_inactivas = int((coincidencias_agrupado["desaparicion_activa"] == False).sum())

print(f"Número de desapariciones activas en PUI es: {comma(num_activas)}\n")
print(f"Número de desapariciones inactivas en PUI es: {comma(num_inactivas)}\n")
print(f"El número de coincidencias únicas en PUI es: {comma(len(coincidencias_agrupado))}")

## Paso 5. Carga de `registros`, `comprimido` y `defunciones`

`registros` son las confrontas manuales: `input/pui/registros_con_pui_DDMMYYYY.xlsx`, con
cinco columnas. **No debe confundirse con `registros_con_pui_activos_*.xlsx`**, que es la
salida de este cuaderno; por eso la búsqueda excluye los nombres que contienen `activos`.

Esas cinco columnas son las que fijan el ancho de la salida en diez: cinco de aquí, cuatro que
aporta la PUI (`indicadora_coincidencia_manual`, `tipo_tramites`, `tramites_repetidos`,
`fecha_evento`) y `clasificacion_des`, que llega desde `comprimido`. Si `registros` trajera
además su propia `clasificacion_des`, el cruce con `comprimido` la duplicaría.

In [ ]:
# Los tres insumos se leen con su mapeo de config: archivo, hoja y columnas
# exigidos. Vuelven con las columnas renombradas al nombre interno del mapeo.

# --- registros: las 5 columnas que fijan el ancho de la salida ---
registros = leer_excel(
    RUTA_REGISTROS,
    config.PUI_REGISTROS_COLUMN_MAPPING,
    hoja=config.PUI_REGISTROS_SHEET,
    descripcion="confrontas manuales (registros)",
)

if registros["fub"].duplicated().any():
    warnings.warn(
        f"`registros` trae {comma(int(registros['fub'].duplicated().sum()))} FUB duplicados; "
        "los cruces por FUB multiplicarán esas filas (mismo comportamiento que left_join en R)."
    )

# --- comprimido: las 4 columnas que este cuaderno necesita del Parquet del notebook 1 ---
# El mapeo resuelve `curp_victima` -> `curp_vic_derivada_rnpdno` (la CURP imputada).
comprimido = leer_parquet(
    RUTA_COMPRIMIDO,
    config.PUI_COMPRIMIDO_COLUMN_MAPPING,
    descripcion="comprimido (RNPDNO del notebook 1)",
)
# Asegurar que la fecha tiene timezone UTC (puede llegar tz-aware del parquet)
fecha_temp = pd.to_datetime(comprimido["fecha_hechos_modificada"], errors="coerce")
if fecha_temp.dt.tz is None:
    comprimido["fecha_hechos_modificada"] = fecha_temp.dt.tz_localize("UTC")
else:
    comprimido["fecha_hechos_modificada"] = fecha_temp

# --- defunciones ---
defunciones = leer_excel(
    RUTA_DEFUNCIONES,
    config.PUI_DEFUNCIONES_COLUMN_MAPPING,
    hoja=config.PUI_DEFUNCIONES_SHEET,
    descripcion="defunciones por institución",
)

print(f"\u2705 registros    : {comma(len(registros))} filas")
print(f"\u2705 comprimido   : {comma(len(comprimido))} filas")
print(f"\u2705 defunciones  : {comma(len(defunciones))} filas")

## Paso 6. Cruce con las confrontas manuales

Primero por **FUB**. Los que no casan se reintentan por **CURP + fecha de desaparición**, para
recuperar los casos que son la misma persona bajo un FUB distinto.

In [ ]:
# --- 6.1 Cruce por FUB ---
registros_manual = (
    registros[registros["indicadora_indicio_de_vida"] == "con registro administrativo"]
      .assign(indicadora_manual="con coincidencia manual")
      .rename(columns={
          "indicadora_indicio_de_vida": "indicadora_indicio_de_vida_manual",
          "institucion_indicadora_de_vida": "institucion_indicadora_de_vida_manual",
          "numero_instituciones": "numero_instituciones_rnpdno_manual",
      })
      [["fub", "indicadora_indicio_de_vida_manual", "institucion_indicadora_de_vida_manual",
        "numero_instituciones_rnpdno_manual", "indicadora_manual"]]
)

coincidencias_agrupado = coincidencias_agrupado.merge(registros_manual, on="fub", how="left")
coincidencias_agrupado["indicadora_manual"] = (
    coincidencias_agrupado["indicadora_manual"].fillna("sin coincidencia manual")
)

sin_manual_fase1 = coincidencias_agrupado[
    coincidencias_agrupado["indicadora_manual"] == "sin coincidencia manual"
].copy()

print(f"Con coincidencia manual por FUB : "
      f"{comma(int((coincidencias_agrupado['indicadora_manual'] == 'con coincidencia manual').sum()))}")
print(f"Sin coincidencia manual por FUB : {comma(len(sin_manual_fase1))}")

In [ ]:
# --- 6.2 Cruce por CURP + fecha de desaparición ---
# `registros` se reasigna aquí, tal como en el script: queda filtrado y con la fecha de hechos.
registros = (
    registros[registros["indicadora_indicio_de_vida"] == "con registro administrativo"]
      .merge(comprimido[["fub", "fecha_hechos_modificada"]], on="fub", how="left")
)
registros["curp_fecha_desaparicion"] = np.where(
    registros["curp_victima"].isna(),
    np.nan,
    registros["curp_victima"].astype(str) + "_"
        + registros["fecha_hechos_modificada"].dt.strftime("%Y-%m-%d"),
)

sin_manual_fase1["curp_fecha_desaparicion"] = np.where(
    sin_manual_fase1["curp"].isna(),
    np.nan,
    sin_manual_fase1["curp"].astype(str) + "_" + sin_manual_fase1["fecha_desaparicion"].astype(str),
)

EN_AMBAS = ("Registro en ambas (coincidencias administrativas sin match manual y "
            "coincidencias manuales) pero distinta FUB")
SOLO_ADMIN = "Registro solo en coincidencias administrativas sin match manual"

llaves_registros = set(registros["curp_fecha_desaparicion"].dropna())
sin_manual_fase1["coincidencia_curp_fecha"] = np.where(
    sin_manual_fase1["curp_fecha_desaparicion"].isin(llaves_registros), EN_AMBAS, SOLO_ADMIN
)

# Los que siguen sin match: sólo PUI.
sin_manual_fase2 = sin_manual_fase1[
    sin_manual_fase1["coincidencia_curp_fecha"] == SOLO_ADMIN
].copy()

# Los que sí casaron, por FUB o por CURP + fecha.
con_manual = pd.concat(
    [
        coincidencias_agrupado[coincidencias_agrupado["indicadora_manual"] == "con coincidencia manual"],
        sin_manual_fase1[sin_manual_fase1["coincidencia_curp_fecha"] == EN_AMBAS],
    ],
    ignore_index=True,
)

print(f"El número de coincidencias únicas en PUI sin match con confrontas manuales "
      f"(por FUB o por curp + fecha de desaparición) es: {comma(len(sin_manual_fase2))}\n")
print(f"El número de coincidencias en PUI que también tienen coincidencias con confrontas "
      f"manuales (por FUB o por curp + fecha de desaparición) es: {comma(len(con_manual))}\n")

In [ ]:
# Instituciones presentes únicamente en las coincidencias de la PUI
instituciones_unicas = sorted(
    sin_manual_fase2["instituciones"]
      .str.split(",")
      .explode()
      .str.strip()
      .dropna()
      .unique()
)

print("Las instituciones únicas que aparecen solamente en las coincidencias con PUI:")
for inst in instituciones_unicas:
    print(f"  - {inst}")
print(f"\nEl número de instituciones únicas que aparecen solamente en las coincidencias "
      f"con PUI es: {len(instituciones_unicas)}\n")

## Paso 7. Abreviaturas y base combinada

Se sustituyen los nombres largos de las instituciones por sus abreviaturas y se unen las dos
ramas —sólo PUI y PUI + manual— en una sola tabla.

In [ ]:
for df in (sin_manual_fase2, con_manual):
    df["instituciones"] = r_str_replace_all(df["instituciones"], ABREVIATURAS_INSTITUCIONES)

sin_manual_fase2["indicadora_coincidencia_manual"] = "solo coincidencia en PUI"
con_manual["indicadora_coincidencia_manual"] = "coincidencia en PUI y registro manual"

combinado = pd.concat([sin_manual_fase2, con_manual], ignore_index=True)

combinado = combinado.drop(columns=[
    "indicadora_indicio_de_vida_manual", "institucion_indicadora_de_vida_manual",
    "numero_instituciones_rnpdno_manual", "indicadora_manual",
    "curp_fecha_desaparicion", "coincidencia_curp_fecha",
], errors="ignore")

combinado = combinado.rename(columns={"instituciones": "institucion_indicadora_de_vida"})

combinado["indicadora_indicio_de_vida"] = np.select(
    [
        combinado["indicadora_coincidencia_manual"] == "coincidencia en PUI y registro manual",
        combinado["indicadora_coincidencia_manual"] == "solo coincidencia en PUI",
    ],
    ["con registro administrativo", "con registro administrativo en PUI"],
    default=None,
)

combinado = combinado.rename(columns={
    "curp": "curp_victima",
    "no_instituciones": "numero_instituciones",
    "tipo_evento": "tipo_tramites",
})[[
    "fub", "curp_victima", "indicadora_indicio_de_vida", "institucion_indicadora_de_vida",
    "numero_instituciones", "indicadora_coincidencia_manual", "tipo_tramites",
    "tramites_repetidos",
    "desaparicion_activa",  # se elimina al final; hasta entonces filtra activos/localizados
    "fecha_evento",         # necesaria para acotar el reporte semanal por fecha de corte
]]

print(f"✅ Base combinada: {comma(len(combinado))} filas.")
print(combinado["indicadora_coincidencia_manual"].value_counts().to_string())

## Paso 8. Pegado con los registros manuales

Los FUB presentes en ambas fuentes acumulan instituciones: se concatenan los nombres y se
suman los conteos. Los que sólo existen en la PUI se añaden aparte, y se descartan los que no
tienen FUB en el RNPDNO.

In [ ]:
# --- 8.1 Registros que están en ambas fuentes ---
registros = registros[registros["indicadora_indicio_de_vida"] == "con registro administrativo"]
# Fuera las defunciones que ya venían en `registros` (llegan con numero_instituciones == 0);
# se reincorporan en el paso 9 desde su propio archivo.
registros = registros[registros["numero_instituciones"] > 0]

lado_pui = (
    combinado[combinado["indicadora_coincidencia_manual"] == "coincidencia en PUI y registro manual"]
      .drop(columns=["curp_victima", "indicadora_indicio_de_vida"])
      .rename(columns={
          "institucion_indicadora_de_vida": "institucion_indicadora_de_vida_pui",
          "numero_instituciones": "numero_instituciones_pui",
      })
)

registros_con_pui = registros.merge(lado_pui, on="fub", how="left")
registros_con_pui = registros_con_pui.drop(columns=["curp_fecha_desaparicion"])

en_ambas = registros_con_pui["indicadora_coincidencia_manual"] == "coincidencia en PUI y registro manual"
registros_con_pui["institucion_indicadora_de_vida"] = np.where(
    en_ambas,
    registros_con_pui["institucion_indicadora_de_vida"].astype(str) + ", "
        + registros_con_pui["institucion_indicadora_de_vida_pui"].astype(str),
    registros_con_pui["institucion_indicadora_de_vida"],
)
registros_con_pui["numero_instituciones"] = np.where(
    en_ambas,
    registros_con_pui["numero_instituciones"] + registros_con_pui["numero_instituciones_pui"],
    registros_con_pui["numero_instituciones"],
)
registros_con_pui = registros_con_pui.drop(
    columns=["institucion_indicadora_de_vida_pui", "numero_instituciones_pui"])

print(f"✅ registros_con_pui (rama manual): {comma(len(registros_con_pui))} filas.")

In [ ]:
# --- 8.2 Registros que sólo existen en la PUI ---
solo_pui = (
    combinado[combinado["indicadora_coincidencia_manual"] == "solo coincidencia en PUI"]
      .merge(
          comprimido[["fub", "clasificacion_des"]].assign(con_fub_en_rnpdno="con fub en rnpdno"),
          on="fub", how="left",
      )
)
solo_pui["con_fub_en_rnpdno"] = solo_pui["con_fub_en_rnpdno"].fillna("sin fub en rnpdno")

n_con_fub = int((solo_pui["con_fub_en_rnpdno"] == "con fub en rnpdno").sum())
n_sin_fub = int((solo_pui["con_fub_en_rnpdno"] == "sin fub en rnpdno").sum())
print(f"El número de registros con solo coincidencia en PUI que tienen FUB en RNPDNO es: {comma(n_con_fub)}\n")
print(f"El número de registros con solo coincidencia en PUI que NO tienen FUB en RNPDNO es: {comma(n_sin_fub)}\n")

print("Descartando los registros con solo coincidencia en PUI que NO tienen FUB en RNPDNO...\n")
solo_pui = (
    solo_pui[solo_pui["con_fub_en_rnpdno"] == "con fub en rnpdno"]
      .drop(columns=["con_fub_en_rnpdno", "clasificacion_des"])
)

registros_con_pui = registros_con_pui.drop(columns=["fecha_hechos_modificada"])
registros_con_pui = pd.concat([registros_con_pui, solo_pui], ignore_index=True)

print(f"✅ registros_con_pui (ambas ramas): {comma(len(registros_con_pui))} filas.")

## Paso 9. Filtro de activos y anexo de defunciones

Se conservan las desapariciones activas y las de la rama manual (`desaparicion_activa` nula,
porque no vienen de la PUI y no se sabe su estado), acotadas a `Desaparecidos` y `Pendientes`.
Después se suman las defunciones posteriores al 1 de enero de 2006.

In [ ]:
# --- 9.1 Activos, acotados a Desaparecidos / Pendientes ---
registros_con_pui_activos = registros_con_pui[
    (registros_con_pui["desaparicion_activa"] == True)
    | (registros_con_pui["desaparicion_activa"].isna())
].copy()

registros_con_pui_activos = registros_con_pui_activos.merge(
    comprimido[["fub", "clasificacion_des"]], on="fub", how="left")
registros_con_pui_activos = registros_con_pui_activos[
    registros_con_pui_activos["clasificacion_des"].isin(["Desaparecidos", "Pendientes"])
]

print(f"El número de registros con coincidencias en PUI y en confrontas manuales que tienen "
      f"clasificación de 'Desaparecidos' o 'Pendientes' en RNPDNO es: "
      f"{comma(len(registros_con_pui_activos))}\n")

In [ ]:
# --- 9.2 Defunciones ---
defunciones_df = defunciones.merge(
    comprimido[["fub", "curp_victima", "clasificacion_des", "fecha_hechos_modificada"]],
    on="fub", how="left",
)
defunciones_df = defunciones_df[
    defunciones_df["clasificacion_des"].isin(["Desaparecidos", "Pendientes"])
]
defunciones_df = defunciones_df[
    defunciones_df["fecha_hechos_modificada"] >= pd.Timestamp("2006-01-01", tz="UTC")
]

defunciones_df["indicadora_indicio_de_vida"] = "con registro administrativo"
defunciones_df = defunciones_df.rename(
    columns={"tipo_defuncion_institucion": "institucion_indicadora_de_vida"})
# "defuncion_INE" → "defuncion de INE"
defunciones_df["institucion_indicadora_de_vida"] = (
    defunciones_df["institucion_indicadora_de_vida"].str.replace("_", " de ", regex=False)
)
# En defunciones no interesa acumular instituciones, sólo marcar que existe el registro.
defunciones_df["numero_instituciones"] = 0

defunciones_df = defunciones_df[[
    "fub", "curp_victima", "indicadora_indicio_de_vida",
    "institucion_indicadora_de_vida", "numero_instituciones", "clasificacion_des",
]]

print(f"El número de registros de defunciones que se van a pegar con los registros de PUI es: "
      f"{comma(len(defunciones_df))}\n")

In [ ]:
# --- 9.3 Unión final y limpieza de tipo_tramites ---
registros_con_pui_activos = pd.concat(
    [registros_con_pui_activos, defunciones_df], ignore_index=True)

print(f"El número total de registros con coincidencias administrativas (tanto defunciones como "
      f"indicios de vida) en PUI y en RNPDNO es: {comma(len(registros_con_pui_activos))}\n")

# `desaparicion_activa` ya cumplió su función: la tabla mezcla activos y defunciones.
registros_con_pui_activos = registros_con_pui_activos.drop(columns=["desaparicion_activa"])

registros_con_pui_activos["tipo_tramites"] = r_str_replace_all(
    registros_con_pui_activos["tipo_tramites"], ABREVIATURAS_INSTITUCIONES)

# El contrato de salida vive en config.PUI_OUTPUT_COLUMNS (ver docs/8_indicios_de_vida.md).
COLS_SALIDA = config.PUI_OUTPUT_COLUMNS
registros_con_pui_activos = registros_con_pui_activos[COLS_SALIDA]

# Tipos: el CSV de la PUI se leyó como texto para no perder los ceros de `institucion_id`,
# así que `fecha_evento` llega como cadena y `numero_instituciones` como float (residuo de
# los cruces). Se restituyen los tipos que tiene el archivo que produce el script de R.
registros_con_pui_activos["fecha_evento"] = pd.to_datetime(
    registros_con_pui_activos["fecha_evento"], errors="coerce")
registros_con_pui_activos["numero_instituciones"] = (
    registros_con_pui_activos["numero_instituciones"].astype("Int64"))

registros_con_pui_activos.head(5)

## Paso 10. Validación y guardado

Comprobaciones de forma antes de escribir: las diez columnas en orden, sin FUB nulos ni
duplicados, y `clasificacion_des` acotada a `Desaparecidos` / `Pendientes`.

In [ ]:
problemas = []

if list(registros_con_pui_activos.columns) != COLS_SALIDA:
    problemas.append(f"El orden de columnas no coincide: {list(registros_con_pui_activos.columns)}")

n_fub_nulos = int(registros_con_pui_activos["fub"].isna().sum())
if n_fub_nulos:
    problemas.append(f"Hay {comma(n_fub_nulos)} FUB nulos.")

# Duplicados de FUB: hay que separar dos casos distintos.
#
#   • Solape PUI ↔ defunciones. La misma persona puede tener un trámite en la PUI y un acta
#     de defunción. Las dos ramas se unen con un bind_rows que no deduplica, así que salen
#     dos filas. El script de R hace exactamente lo mismo: el filtro `numero_instituciones > 0`
#     limpia las defunciones de `registros`, pero no toca la rama "solo coincidencia en PUI".
#     Es esperado, no un defecto.
#
#   • Duplicados dentro de una misma rama. Eso sí es anómalo y apunta a FUB repetidos en
#     alguno de los insumos.
es_defuncion = registros_con_pui_activos["institucion_indicadora_de_vida"].str.startswith(
    "defuncion de", na=False)
dup_en_rama = int(registros_con_pui_activos.loc[~es_defuncion, "fub"].duplicated().sum()) + \
              int(registros_con_pui_activos.loc[es_defuncion, "fub"].duplicated().sum())
solape = len(set(registros_con_pui_activos.loc[es_defuncion, "fub"])
             & set(registros_con_pui_activos.loc[~es_defuncion, "fub"]))

if dup_en_rama:
    problemas.append(f"Hay {comma(dup_en_rama)} FUB duplicados dentro de una misma rama.")
if solape:
    print(f"ℹ️  {comma(solape)} FUB aparecen tanto en la PUI como en defunciones y ocupan dos "
          f"filas cada uno. Es el comportamiento del script de R, no un error.\n")

clases = set(registros_con_pui_activos["clasificacion_des"].dropna().unique())
if not clases <= {"Desaparecidos", "Pendientes"}:
    problemas.append(f"clasificacion_des trae valores inesperados: {clases - {'Desaparecidos', 'Pendientes'}}")

if problemas:
    for p in problemas:
        warnings.warn(p)
else:
    print("✅ Validaciones de forma superadas.\n")

print(f"Filas: {comma(len(registros_con_pui_activos))}   Columnas: {registros_con_pui_activos.shape[1]}\n")
print("indicadora_indicio_de_vida:")
print(registros_con_pui_activos["indicadora_indicio_de_vida"].value_counts(dropna=False).to_string())
print("\nindicadora_coincidencia_manual (los nulos son defunciones y confrontas manuales sin PUI):")
print(registros_con_pui_activos["indicadora_coincidencia_manual"].value_counts(dropna=False).to_string())
print("\nclasificacion_des:")
print(registros_con_pui_activos["clasificacion_des"].value_counts(dropna=False).to_string())

In [ ]:
# Se exporta con los nombres de config.PUI_NOMBRES_EXPORTADOS (sufijo de origen). En
# memoria siguen los del script de R, que usa la comparación del paso 11.
registros_con_pui_activos.rename(columns=config.PUI_NOMBRES_EXPORTADOS).to_parquet(
    RUTA_SALIDA, index=False)

print(f"✅ Archivo de salida del ETL oficial guardado: {RUTA_SALIDA}")
print(f"   {comma(len(registros_con_pui_activos))} filas × {registros_con_pui_activos.shape[1]} columnas")
print(f"   corte de los datos: {FECHA_CORTE}")
print(f"\nEl pipeline principal lee la salida autónoma configurada en CONFRONTAS_FILE:")
print(f"   {config.CONFRONTAS_FILE}")

## Paso 11 (opcional). Comparación contra el archivo de R

Si en `input/confrontas/` existe el archivo del mismo corte generado por el script de R, esta
celda contrasta ambos: cuántos FUB comparten, cuáles sobran o faltan, y en qué columnas
discrepan las filas comunes. Si no existe, la celda no hace nada.

In [ ]:
# Archivo de R con el que contrastar, declarado en config.PUI_REFERENCIA_FILE.
# None = no hay con qué comparar y la celda no hace nada. Es la única entrada
# OPCIONAL del cuaderno: no es un insumo del cálculo, sino una verificación.
ruta_referencia = getattr(config, "PUI_REFERENCIA_FILE", None)

if ruta_referencia is None:
    print("ℹ️  config.PUI_REFERENCIA_FILE está en None; se omite la comparación.")
elif not Path(ruta_referencia).exists():
    raise InsumoInvalido(
        f"❌ config.PUI_REFERENCIA_FILE apunta a un archivo que no existe:\n"
        f"   {ruta_referencia}\n"
        f"   Corrige la ruta o ponla en None para omitir la comparación."
    )
else:
    ruta_referencia = Path(ruta_referencia)
    ref = pd.read_excel(ruta_referencia)
    print(f"Referencia: {ruta_referencia.name}  →  {comma(len(ref))} filas")
    print(f"Cuaderno  : {RUTA_SALIDA.name}  →  {comma(len(registros_con_pui_activos))} filas\n")

    if list(ref.columns) != COLS_SALIDA:
        print(f"⚠️  Las columnas difieren.\n   referencia: {list(ref.columns)}")
    else:
        print("✅ Mismas columnas, en el mismo orden.\n")

    fubs_ref = set(ref["fub"])
    fubs_mio = set(registros_con_pui_activos["fub"])
    print(f"FUB en ambos            : {comma(len(fubs_ref & fubs_mio))}")
    print(f"FUB sólo en el cuaderno : {comma(len(fubs_mio - fubs_ref))}")
    print(f"FUB sólo en referencia  : {comma(len(fubs_ref - fubs_mio))}\n")

    # Discrepancias columna por columna sobre los FUB comunes (una fila por FUB en cada lado).
    a = registros_con_pui_activos.drop_duplicates("fub").set_index("fub").sort_index()
    b = ref.drop_duplicates("fub").set_index("fub").sort_index()
    comunes = sorted(fubs_ref & fubs_mio)
    a, b = a.loc[comunes], b.loc[comunes]

    print("Filas comunes con valor distinto, por columna:")
    for col in COLS_SALIDA[1:]:
        izq, der = a[col], b[col]
        if pd.api.types.is_numeric_dtype(izq) and pd.api.types.is_numeric_dtype(der):
            distintos = ~((izq == der) | (izq.isna() & der.isna()))
        else:
            distintos = ~((izq.astype(str) == der.astype(str)) | (izq.isna() & der.isna()))
        n = int(distintos.sum())
        marca = "✅" if n == 0 else "⚠️ "
        print(f"  {marca} {col:32s} {comma(n)}")